# 固定执行目标轨迹的成本反事实
预先固定候选0bp目标与HOLD日历，不选择参考轨迹。三套完整结算账户，0bp必须逐日复现原账户。只作诊断，不作为可部署的风险策略。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.backtest.frozen_open_trace import FrozenOpenTrace
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/frozen_open_cost_20261015';parent=root/'research/opening_state_training_20261015';plan=json.loads((card/'plan.json').read_text());p=plan['account_plan'];provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
source,audit=_source(root,plan['source'],[f'analysis/native/candidate_{v}' for v in plan['costs']]+[f'analysis/native/soxx_{v}' for v in plan['costs']])
market,market_audit=_source(root,plan['market_source'],['inputs/frame']);frame=market['inputs/frame'];frame=frame.loc[frame.signal_date.between(p['start'],p['end'])].copy()
anchor=source['analysis/native/candidate_0'];policy=FrozenOpenTrace(anchor['targets'],anchor['authority']);books={};rows=[]
for slip in plan['costs']:
 replay=replay_full_path(frame,models=[policy],plan=p,slippage_bps=slip)
 book=native_tiered_stage1(frame,replay,provider_uri=provider,tariff=p['tariff'],slippage_bps=slip,initial_cash=p['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition)
 books[slip]=book
 if slip==0:
  columns=['strategy_return','equity','ending_soxl_quantity','ending_soxx_quantity','ending_cash','broker_and_external_fees_usd','slippage_reserve_usd']
  np.testing.assert_allclose(book['authority'][columns],anchor['authority'][columns],rtol=0,atol=1e-8)
  assert len(book['orders'])==len(anchor['orders'])
 independent=source[f'analysis/native/candidate_{slip}']['authority'];path=book['authority'];base=anchor['authority'];benchmark=source[f'analysis/native/soxx_{slip}']['authority']
 mechanical=np.log(path.equity.iloc[-1]/base.equity.iloc[-1]);adaptive=np.log(independent.equity.iloc[-1]/path.equity.iloc[-1]);total=np.log(independent.equity.iloc[-1]/base.equity.iloc[-1]);assert abs(mechanical+adaptive-total)<1e-12
 rows.append({'slippage_bps':slip,**paired_metrics(path,benchmark),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'independent_return_pct':100*(independent.equity.iloc[-1]-1),'fixed_trace_log_effect':mechanical,'adaptive_path_log_residual':adaptive,'total_log_effect':total,'orders':len(book['orders']),'broker_fees_usd':path.broker_and_external_fees_usd.sum(),'slippage_usd':path.slippage_reserve_usd.sum()})
table=pd.DataFrame(rows);policy_meta={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['analysis/native/*','inputs/*']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit,market_audit=market_audit,anchor_targets=anchor['targets'],anchor_ledger=anchor['authority'],policy_code=(root/'src/backtest/frozen_open_trace.py').read_text());rec.save_objects(artifact_path='analysis',comparison=table)
 for slip,book in books.items():rec.save_objects(artifact_path='analysis/native',**{str(slip):book})
 rec.log_metrics(new_native_cases=3,zero_bps_identity_passed=1,registered_artifact_count=10)
summary={'experiment':card.name,'decision':'diagnostic_only_not_promotable','new_fits':0,'new_native_cases':3,'zero_bps_identity_passed':True,'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy_meta,'scope':plan['scope']}
table.to_csv(card/'cost_decomposition.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy_meta}))
binding=json.loads((parent/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2))
print(table[['slippage_bps','return_pct','monitored_drawdown_pct','fixed_trace_log_effect','adaptive_path_log_residual']].to_string(index=False))


# 固定执行轨迹成本反事实：费用损耗与路径变化已分开

完成0拟合、3套新增Qlib原生回测。预先指定上一轮开盘候选0bp账户的目标权重和执行/HOLD日历，不按结果选择参考成本。源账户无恢复时段交易；有恢复交易则拒绝运行，不能静默遗漏。仍以10万美元、同一IBKR Pro Tiered投影、真实历史研究成交标记和整股订单重结算费用、股数与现金。

0bp固定轨迹逐日复现源账户：收益、净值、SOXL/SOXX数量、现金和费用绝对误差均不超过1e-8，订单数量一致。三个账户分别经原生Qlib对账，未用扣一个总费用数代替完整结算。

| 滑点 | 固定轨迹净收益 | 独立风险账户净收益 | 固定轨迹监控最大回撤 |
|---|---:|---:|---:|
| 0 bp | +102.24% | +102.24% | 31.00% |
| 2 bp | +99.92% | +139.55% | 31.22% |
| 5 bp | +96.66% | +118.44% | 31.53% |

以0bp为锚，2bp固定轨迹对数财富影响-0.011521，独立路径相对固定轨迹的对数残差+0.180842，两者之和+0.169321等于原独立账户的总差。5bp分别为-0.027937、+0.105009，总计+0.077072。逐情景恒等式通过1e-12核对。该分解支持：此候选的更高滑点终值改善由不同执行/风控路径驱动，费用及其机械结算反馈本身损害了固定轨迹财富。

固定的是目标比例和动作日期，不是固定股数；净值变化、整股取整、阶梯累计股数与后续费用变化仍包含在固定轨迹效应内。因此不将其叫作纯静态费率扣除。残差包含所有自主路径响应，不能单独归给12%或22%某一阈值，也不证明取消风险控制有益。冻结原风险决策不适合新费用下的真实自身净值，不能作为可部署策略或替代独立账户门禁。

候选固定轨迹在三个成本下仍违反30%回撤要求，且远落后SOXX。此诊断不产生新候选、不晋升，生产不变。成本反事实路线已完成，不更换锚点或扫阈值以制造赢家；此前停止周期重启和风险旁路实验的失败结论仍有效。

下一研究必须检验完整策略的结构性增量，例如先核查冻结执行路由在新配置模型上的适用性，而不能把本轮路径运气当成新信息或继续只优化费率。需要先确认该机制尚未在既有固定实验中测试，再冻结唯一干预及同风险对照。稳定净SOXX超额目标仍未完成。

执行notebook、原生账户和来源证据均在Qlib Recorder，NAS逐对象冷恢复哈希通过。新增固定轨迹HOLD/日期/恢复交易合同测试1项通过。
